In [2]:
import kagglehub
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("austinreese/craigslist-carstrucks-data")
print("Path to dataset files:", path)

cars = pd.read_csv(f"{path}/vehicles.csv")
print(cars.shape)
print(cars.columns.tolist())
cars.head()

100%|██████████| 262M/262M [05:09<00:00, 889kB/s]  

Extracting files...


Path to dataset files: /Users/abaid/.cache/kagglehub/datasets/austinreese/craigslist-carstrucks-data/versions/10
(426880, 26)
['id', 'url', 'region', 'region_url', 'price', 'year', 'manufacturer', 'model', 'condition', 'cylinders', 'fuel', 'odometer', 'title_status', 'transmission', 'VIN', 'drive', 'size', 'type', 'paint_color', 'image_url', 'description', 'county', 'state', 'lat', 'long', 'posting_date']


,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,...,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
0,7222695916,https://prescott.craigslist.org/cto/d/prescott...,prescott,https://prescott.craigslist.org,6000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN
1,7218891961,https://fayar.craigslist.org/ctd/d/bentonville...,fayetteville,https://fayar.craigslist.org,11900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN
2,7221797935,https://keys.craigslist.org/cto/d/summerland-k...,florida keys,https://keys.craigslist.org,21000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN
3,7222270760,https://worcester.craigslist.org/cto/d/west-br...,worcester / central MA,https://worcester.craigslist.org,1500,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN,NaN
4,7210384030,https://greensboro.craigslist.org/cto/d/trinit...,greensboro,https://greensboro.craigslist.org,4900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN,NaN


In [3]:
print(cars["description"].isna().sum(), "missing descriptions")
print(cars["description"].str.split().str.len().describe())
cars[["type", "fuel", "transmission", "drive", "condition"]].nunique()

70 missing descriptions
count    426810.000000
mean        433.005904
std         419.547700
min           1.000000
25%          92.000000
50%         357.000000
75%         654.000000
max        4777.000000
Name: description, dtype: float64


type            13
fuel             5
transmission     3
drive            3
condition        6
dtype: int64

In [4]:
# actual category values, not just counts
for col in ["type", "fuel", "transmission", "drive", "condition"]:
    print(f"--- {col} ---")
    print(cars[col].value_counts(dropna=False))
    print()

--- type ---
type
NaN            92858
sedan          87056
SUV            77284
pickup         43510
truck          35279
other          22110
coupe          19204
hatchback      16598
wagon          10751
van             8548
convertible     7731
mini-van        4825
offroad          609
bus              517
Name: count, dtype: int64

--- fuel ---
fuel
gas         356209
other        30728
diesel       30062
hybrid        5170
NaN           3013
electric      1698
Name: count, dtype: int64

--- transmission ---
transmission
automatic    336524
other         62682
manual        25118
NaN            2556
Name: count, dtype: int64

--- drive ---
drive
4wd    131904
NaN    130567
fwd    105517
rwd     58892
Name: count, dtype: int64

--- condition ---
condition
NaN          174104
good         121456
excellent    101467
like new      21178
fair           6769
new            1305
salvage         601
Name: count, dtype: int64



In [5]:
# this dataset is notorious for duplicate/reposted listings across regions
print("duplicate descriptions:", cars["description"].duplicated().sum())
print("duplicate VIN (non-null only):", cars.loc[cars["VIN"].notna(), "VIN"].duplicated().sum())

duplicate descriptions: 65968
duplicate VIN (non-null only): 147574


In [6]:
# eyeball a few short ones and a few very long ones to judge signal vs. boilerplate
word_counts = cars["description"].str.split().str.len()

print("--- SHORT (5-15 words) ---")
for d in cars.loc[word_counts.between(5, 15), "description"].sample(3, random_state=1):
    print(d, "\n")

print("--- LONG (2000+ words) ---")
for d in cars.loc[word_counts >= 2000, "description"].sample(2, random_state=1):
    print(d[:800], "...\n")

--- SHORT (5-15 words) ---
Very beautiful car for sale 

LOADED! NAVIGATION! MOONROOF! RARE INTERIOR! 4WD! WARRANTY! NEW INSPECTION! 

LT white 4cyl auto 4door fully loaded stereo cd alloys runs great 134k call 518-573-6414 

--- LONG (2000+ words) ---
2017 *** BMW 5 Series 530i 4dr Sedan Sedan ***    CALL/TEXT JASMINE (916) 586-6111Call ☏ (916) 586−6111  Sacramento Luxury Motors 7801 Fair Oaks Blvd, Carmichael, CA 95608Copy & Paste the URL belowto view more information!http://sacramentoluxurymotors.v12soft.com/cars/13451361    			Year : 2017				Make : BMW				Model : 5 Series				Trim : 530i 4dr Sedan				   Mileage : 46,345 miles				Transmission : Automatic				Exterior Color : White				Interior Color : Black				Series : 530i 4dr Sedan Sedan				Drivetrain : RWD				Condition : Excellent				VIN : WBAJA5C30HG895266				Stock ID : 14015				Engine : 2.0L I4	   	Description of this BMW 5 Series 530i 4dr Sedan 	 	WWW.SACRAMENTOLUXURYMOTORS.COM Call us @ 916 944 1480 Text JASMIN @ 916 586 6111 78

In [7]:
cars[["price", "odometer", "year"]].describe()

,price,odometer,year
count,4.268800e+05,4.224800e+05,425675.000000
mean,7.519903e+04,9.804333e+04,2011.235191
std,1.218228e+07,2.138815e+05,9.452120
min,0.000000e+00,0.000000e+00,1900.000000
25%,5.900000e+03,3.770400e+04,2008.000000
50%,1.395000e+04,8.554800e+04,2013.000000
75%,2.648575e+04,1.335425e+05,2017.000000
max,3.736929e+09,1.000000e+07,2022.000000


In [8]:
import re

# drop missing descriptions
cars = cars[cars["description"].notna()].copy()

# drop exact duplicate ads (dealer reposts across regions)
cars = cars.drop_duplicates(subset="description", keep="first")

# word-count filter: cuts near-empty ads AND spam-length boilerplate
word_count = cars["description"].str.split().str.len()
cars = cars[word_count.between(15, 200)]

# realistic value ranges — drops scam/typo entries at the extremes
cars = cars[cars["price"].between(500, 100_000)]
cars = cars[cars["odometer"].between(500, 300_000)]
cars = cars[cars["year"].between(1990, 2022)]

print(cars.shape)

(108390, 26)


In [9]:
def clean_description(text):
    text = re.sub(r'http\S+|www\.\S+', ' ', text)                      # URLs
    text = re.sub(r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', ' ', text)    # phone numbers
    text = re.sub(r'[☏✅🚘📞]+', ' ', text)                              # symbols seen in your samples
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cars["description_clean"] = cars["description"].apply(clean_description)

# sanity check after cleaning
print(cars.shape)
print(cars["description_clean"].str.split().str.len().describe())
for d in cars["description_clean"].sample(5, random_state=1):
    print(d, "\n")

(108390, 27)
count    108390.000000
mean         76.194898
std          48.252887
min          11.000000
25%          38.000000
50%          62.000000
75%         106.000000
max         201.000000
Name: description_clean, dtype: float64
2003 Honda Accord with 166k everything works except radio,runs great ,fast car black leather , sunroof ,remote entry , nice 18 inch wheels show contact info 

88,261 original mile hard to find 2008 Ford F-350 XLT 4wd with the 84” cab to axle. It is ready for a 10’-12’ flatbed or service body. It was a locally owned and is ready for your use. Please call/text 

Selling our 2014 Dodge Grand Caravan. Excellent condition. Looking to downsize to smaller vehicle. Call or text ninezerosix 360-twofoureighttwo 

For Sale: 2014 Ford Police Interceptor Sedan 3.7L AWD. Good condition, Heated seats, no dents, no rust. Drives nice, no issues, 1,000 miles on new firestone firehawk GT pursuit tires.Replaced engine with 3.7L from a 2013 Explorer that has 65,000 miles on

In [10]:
# how much missingness survived the filters?
print(cars[["type","fuel","transmission","drive","condition"]].isna().sum())
print("remaining duplicate VINs:", cars.loc[cars["VIN"].notna(), "VIN"].duplicated().sum())

type            34568
fuel              314
transmission      109
drive           31394
condition       28358
dtype: int64
remaining duplicate VINs: 4056


In [11]:
# bucket missing body type rather than dropping ~20% of rows
cars["type"] = cars["type"].fillna("unknown")

# stratified sample: up to 1000 per body type, keeps variety for the demo
cars_sampled = (
    cars.groupby("type", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 1000), random_state=42))
    .reset_index(drop=True)
)

print(cars_sampled.shape)
print(cars_sampled["type"].value_counts())

(12429, 26)


KeyError: 'type'

In [12]:
import pandas as pd
print(pd.__version__)

3.0.5


In [13]:
import numpy as np

sampled_idx = (
    cars.groupby("type")
    .apply(lambda g: g.sample(n=min(len(g), 1000), random_state=42).index)
)
all_idx = np.concatenate(sampled_idx.values)

cars_sampled = cars.loc[all_idx].reset_index(drop=True)

print(cars_sampled.shape)
print(cars_sampled["type"].value_counts())

(12429, 27)
type
SUV            1000
convertible    1000
coupe          1000
hatchback      1000
mini-van       1000
pickup         1000
sedan          1000
truck          1000
unknown        1000
van            1000
wagon          1000
other           914
offroad         311
bus             204
Name: count, dtype: int64


In [14]:
cars_sampled["tagged_description"] = cars_sampled["id"].astype(str) + " " + cars_sampled["description_clean"]

cars_sampled.to_csv("cars_cleaned.csv", index=False)
print(cars_sampled.shape)
cars_sampled[["id", "type", "manufacturer", "model", "price", "tagged_description"]].head()

(12429, 28)


,id,type,manufacturer,model,price,tagged_description
0,7314068693,SUV,jeep,wrangler,14699,7314068693 2008 Jeep Wrangler X Sport Utility ...
1,7311964042,SUV,NaN,HUMMER H2,22997,7311964042 2003 HUMMER H2 Adventure Series 4dr...
2,7305257417,SUV,chevrolet,trailblazer ls,5500,7305257417 Asking $5500 obo all offers welcome...
3,7311037499,SUV,honda,pilot ex-l,10995,"7311037499 2013 Honda Pilot EX-L, 3.5L V6, Aut..."
4,7315419164,SUV,cadillac,escalade,30777,7315419164 2013 Cadillac Escalade Premium 4dr ...


In [15]:
for col in ["type", "fuel", "transmission", "drive", "condition"]:
    print(f"--- {col} ---")
    print(cars_sampled[col].value_counts(dropna=False))
    print()

print("unique manufacturers:", cars_sampled["manufacturer"].nunique())
print(cars_sampled["price"].describe())

--- type ---
type
SUV            1000
convertible    1000
coupe          1000
hatchback      1000
mini-van       1000
pickup         1000
sedan          1000
truck          1000
unknown        1000
van            1000
wagon          1000
other           914
offroad         311
bus             204
Name: count, dtype: int64

--- fuel ---
fuel
gas         11260
diesel        806
hybrid        175
other         113
NaN            38
electric       37
Name: count, dtype: int64

--- transmission ---
transmission
automatic    10807
manual        1418
other          187
NaN             17
Name: count, dtype: int64

--- drive ---
drive
fwd    3986
4wd    3423
rwd    2771
NaN    2249
Name: count, dtype: int64

--- condition ---
condition
excellent    4400
good         3840
NaN          2178
like new     1309
fair          606
salvage        48
new            48
Name: count, dtype: int64

unique manufacturers: 38
count    12429.000000
mean     12688.527878
std      11306.933675
min        500.000

In [16]:
for col in ["transmission", "drive", "condition"]:
    print(f"--- {col} ---")
    print(cars_sampled[col].value_counts(dropna=False))
    print()

print("unique manufacturers:", cars_sampled["manufacturer"].nunique())
print(cars_sampled["manufacturer"].value_counts(dropna=False).head(15))

--- transmission ---
transmission
automatic    10807
manual        1418
other          187
NaN             17
Name: count, dtype: int64

--- drive ---
drive
fwd    3986
4wd    3423
rwd    2771
NaN    2249
Name: count, dtype: int64

--- condition ---
condition
excellent    4400
good         3840
NaN          2178
like new     1309
fair          606
salvage        48
new            48
Name: count, dtype: int64

unique manufacturers: 38
manufacturer
ford             2338
chevrolet        1427
toyota           1058
honda             780
NaN               698
dodge             557
subaru            526
jeep              524
nissan            482
ram               380
bmw               366
chrysler          362
gmc               323
mercedes-benz     312
volkswagen        272
Name: count, dtype: int64


In [17]:
print("unique manufacturers:", cars_sampled["manufacturer"].nunique())
print(cars_sampled["manufacturer"].value_counts(dropna=False).head(15))

unique manufacturers: 38
manufacturer
ford             2338
chevrolet        1427
toyota           1058
honda             780
NaN               698
dodge             557
subaru            526
jeep              524
nissan            482
ram               380
bmw               366
chrysler          362
gmc               323
mercedes-benz     312
volkswagen        272
Name: count, dtype: int64


In [18]:
import pandas as pd
cars = pd.read_csv("cars_cleaned.csv")

import torch
from transformers import pipeline

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

vibe_labels = ["family car", "sports car", "luxury car", "eco-friendly car", "off-road vehicle", "daily commuter car"]

Using device: mps


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

In [ ]:
import time

sample = cars["description_clean"].sample(20, random_state=1).tolist()

start = time.time()
results = classifier(sample, vibe_labels, multi_label=True)
elapsed = time.time() - start

print(f"{elapsed:.1f}s for 20 rows -> ~{elapsed/20*12429/60:.1f} min estimated for full dataset")

# eyeball a couple to judge label quality
for text, res in zip(sample[:3], results[:3]):
    print(text[:150])
    print(dict(zip(res["labels"], [round(s,2) for s in res["scores"]])))
    print()